# Use cases

Three concrete lookups, run against real data:

1. **per compound** — all targets it has been tested against
2. **per target** — all compounds tested against it
3. **per protein family** — which compounds are available for it

`probe.db` is now built from `staging/use_cases`, a real ChEMBL-derived
bioactivity dump (not the toy `staging/_template` this notebook started on).
Rebuild it with:

```bash
uv run python examples/populate_db.py
```

That loads every non-underscore directory under `staging/` (currently just
`use_cases`) into a fresh `probe.db`. Delete the existing file first for a
clean rebuild -- `ProbeDB(..., create=True)` refuses to run against a database
that already has tables.

Real data is messier than the toy set, in ways that shape the code below:

- **Coverage is sparse.** 4185 compounds are on file, but only 2 of them have
  any measurement at all -- everything else is unscreened. Looping over every
  compound or every target the way the toy version did would mostly print
  nothing, so both use cases below iterate over what actually has data.
- **Units and endpoint types are all over the place.** IC50, Ki, AC50, Potency,
  radioligand Ka, plus a long tail of ADME and safety endpoints (Cmax, ALT,
  lung weight, hepatotoxicity flags, LogP...) all sit in the same
  `bioactivity` table with units like `nM`, `uM`, `%`, `hr`, or `unspecified`.
  "Main target" and "most potent" below pick a scale dynamically per query
  instead of assuming IC50/nM, and restrict it to a short list of recognized
  potency endpoints so a hepatotoxicity percentage never gets ranked as if it
  were a binding affinity.
- **Target names are not all clean protein names.** Bulk ChEMBL exports carry
  through assay organisms and metadata placeholders as if they were targets --
  `Homo sapiens`, `Mus musculus`, `ADMET`, `No relevant target`, `Unchecked`.
  These are shown as-is rather than filtered out, because filtering them
  quietly would be claiming a cleanup this notebook does not actually do.

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB_PATH = Path("..") / "probe.db"
assert DB_PATH.exists(), f"{DB_PATH} not found -- run examples/populate_db.py first"

db = ProbeDB(DB_PATH, create=False)

db.counts()

,table,rows
0,compound,4185
1,chembl,3053
2,uniprot,2031
3,target,2281
4,target_uniprot,2031
5,bioactivity_source,98
6,bioactivity_group,246
7,bioactivity,909


## Use case 1: per compound

For every compound that has been measured at all, answer four questions:

- **which targets** has it been measured against?
- **which set(s)** do the measurements come from (`source_db`)?
- **what is its main target** — the one with the strongest reported potency?
- **what is its selectivity** — how much weaker is the next best target on the
  same scale?

"Main target" and "selectivity" only compare rows that are actually
comparable: `relation == "="`, a real unit (not `unspecified`), and a
`bioactivity_type` from a short allow-list of recognized potency endpoints --
`IC50`, `EC50`, `AC50`, `Ki`, `Kd`, `GI50`, `CC50`, `ED50`, `Potency`. Among
those, the scale is picked dynamically as whichever `(bioactivity_type, unit)`
pair has the most rows for that compound, rather than assuming IC50 in nM the
way the toy dataset allowed -- one compound's richest scale here is `Potency`
in nM, the other's is `IC50` in nM. Anything measured as a bound (`>`, `<`,
...), or reported outside that allow-list (ADME, safety, PK endpoints), is
never pooled into the ranking; bounded rows are shown separately as
qualitative counter-screens/censored values instead.

In [2]:
POTENCY_TYPES = {"IC50", "EC50", "AC50", "Ki", "Kd", "GI50", "CC50", "ED50", "Potency"}
TOP_N = 10  # cap how many rows get printed per section


def compound_label(db, inchikey):
    # this source leaves compound.name empty, so fall back to a ChEMBL id
    # and finally the InChIKey itself -- always something identifiable
    name = db.one("SELECT name FROM compound WHERE inchikey = ?", inchikey)
    chembl_id = db.one("SELECT chembl_id FROM chembl WHERE inchikey = ?", inchikey)
    return name or chembl_id or inchikey


def numeric_potency_rows(hits):
    return hits[
        hits.bioactivity_type.isin(POTENCY_TYPES)
        & (hits.relation == "=")
        & hits.value.notna()
        & (hits.unit != "unspecified")
    ]


def compound_profile(db, compound):
    hits = db.bioactivities(compound=compound)

    targets = hits[["target_type", "target"]].drop_duplicates().reset_index(drop=True)
    sources = sorted(hits["source_db"].dropna().unique())

    numeric = numeric_potency_rows(hits)
    scale, potency = None, pd.DataFrame(columns=["target", "target_type", "value"])
    if not numeric.empty:
        scale = numeric.groupby(["bioactivity_type", "unit"]).size().idxmax()
        comparable = numeric[
            (numeric.bioactivity_type == scale[0]) & (numeric.unit == scale[1])
        ]
        potency = (
            comparable.groupby(["target", "target_type"], as_index=False)["value"]
            .median()
            .sort_values("value")
            .reset_index(drop=True)
        )

    counter_screens = hits[hits.relation.isin([">", ">=", "<", "<="])]

    return targets, sources, scale, potency, counter_screens


profiled = sorted(db.table("bioactivity")["inchikey"].unique())
print(f"{len(profiled)} of {len(db.table('compound'))} compounds have any bioactivity on file\n")

for inchikey in profiled:
    label = compound_label(db, inchikey)
    targets, sources, scale, potency, counter_screens = compound_profile(db, inchikey)

    print(f"== {label} ==")
    print(f"targets measured: {len(targets)} (showing up to {TOP_N})")
    print(targets.head(TOP_N).to_string(index=False))

    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("main target: no recognized potency measurement "
              "(IC50/EC50/Ki/Kd/AC50/Potency/...) on a named scale")
    else:
        btype, unit = scale
        best = potency.iloc[0]
        print(f"main target ({btype}, {unit}, {len(potency)} targets on this scale): "
              f"{best.target}  ({best.value:g} {unit})")
        if len(potency) > 1:
            second = potency.iloc[1]
            fold = second.value / best.value
            print(
                f"selectivity: {fold:.1f}-fold vs {second.target} "
                f"({second.value:g} {unit}), the next best on the same scale"
            )
        else:
            print(f"selectivity: only one target with comparable {btype} ({unit}) data")

    if not counter_screens.empty:
        print(f"bounded/censored measurements: {len(counter_screens)} rows on other scales "
              f"(showing up to {TOP_N}), read qualitatively:")
        print(
            counter_screens[["target", "bioactivity_type", "relation", "value", "unit"]]
            .head(TOP_N)
            .to_string(index=False)
        )

    print()

2 of 4185 compounds have any bioactivity on file

== CHEMBL100014 ==
targets measured: 12 (showing up to 10)
target_type             target
    protein            B16-F10
    protein              HL-60
    protein               WEHI
    protein          Unchecked
    protein                B16
    protein            3LLD122
    protein         MIA PaCa-2
    protein             MES-SA
    protein NON-PROTEIN TARGET
    protein               PC-3
derived from: ChEMBL
main target (IC50, nM, 9 targets on this scale): MIA PaCa-2  (30000 nM)
selectivity: 1.2-fold vs 3LLD122 (35000 nM), the next best on the same scale
bounded/censored measurements: 4 rows on other scales (showing up to 10), read qualitatively:
    target    bioactivity_type relation      value        unit
   B16-F10         Lung weight        >     0.2686           g
   B16-F10 No. of lung lesions        >   140.2900 unspecified
SARS-CoV-2                IC50        > 20000.0000          nM
SARS-CoV-2                IC50    

## Use case 2: per target

The mirror image of use case 1. For a given target, answer:

- **how many compounds** have been tested against it, and **which set(s)**
  do they come from?
- **which compound is most potent** — on whichever recognized potency scale
  has data for this target?
- **which compound is most selective** — the one for which this target is the
  strongest hit by the widest margin, compared to that same compound's own
  next best target?

"Most selective" reuses `compound_profile` from use case 1: for each compound
tested here, it looks at *that compound's* full potency ranking across all its
targets and asks how this target compares to the compound's best *other*
target. A ratio above 1 means the compound genuinely prefers this target; a
ratio below 1 means even the best candidate here is actually more potent
somewhere else, so nothing tested is truly selective for it.

Only 243 of 2281 targets have any measurement, and only 3 of those have more
than one compound tested -- looping over all 2281 the way the toy version did
would be almost entirely empty output. So this shows the coverage distribution
first, then the full per-target breakdown only for the targets that actually
carry data worth looking at: the busiest ones plus every one with more than
one compound.

In [3]:
def target_profile(db, target_id):
    hits = db.bioactivities(target=target_id)

    compound_keys = sorted(hits["inchikey"].unique())
    sources = sorted(hits["source_db"].dropna().unique())

    numeric = numeric_potency_rows(hits)
    scale, potency = None, pd.DataFrame(columns=["inchikey", "value"])
    if not numeric.empty:
        scale = numeric.groupby(["bioactivity_type", "unit"]).size().idxmax()
        comparable = numeric[
            (numeric.bioactivity_type == scale[0]) & (numeric.unit == scale[1])
        ]
        potency = (
            comparable.groupby("inchikey", as_index=False)["value"]
            .median()
            .sort_values("value")
            .reset_index(drop=True)
        )

    return compound_keys, sources, scale, potency


def target_preference(db, compound, target_name):
    # how this target compares to `compound`'s own best *other* target,
    # reusing the per-compound potency ranking from use case 1
    _, _, _, potency, _ = compound_profile(db, compound)
    at_target = potency[potency.target == target_name]
    others = potency[potency.target != target_name]
    if at_target.empty or others.empty:
        return None
    return others.value.min() / at_target.value.iloc[0]


targeted = db.table("bioactivity").groupby("target_id")["inchikey"].nunique()
row_counts = db.table("bioactivity").groupby("target_id").size()

print(f"{len(targeted)} of {len(db.table('target'))} targets have at least one measurement")
print(targeted.value_counts().sort_index().rename_axis("compounds tested").rename("targets"))
print()

shared = set(targeted[targeted > 1].index)
richest = set(row_counts.sort_values(ascending=False).head(TOP_N).index)
featured = sorted(shared | richest, key=lambda t: -row_counts[t])

print(f"showing {len(featured)} targets: the {len(richest)} with the most measurements on file, "
      f"plus {len(shared - richest)} more that have more than one compound tested\n")

targets_tbl = db.table("target").set_index("target_id")

for target_id in featured:
    target = targets_tbl.loc[int(target_id)]
    compound_keys, sources, scale, potency = target_profile(db, int(target_id))

    print(f"== {target['name']} ({target.type}) ==")
    print(f"compounds tested ({len(compound_keys)}): "
          f"{', '.join(compound_label(db, k) for k in compound_keys)}")
    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("most potent: no recognized potency measurement for this target")
    else:
        btype, unit = scale
        best = potency.iloc[0]
        print(f"most potent ({btype}, {unit}): "
              f"{compound_label(db, best.inchikey)}  ({best.value:g} {unit})")

        ratios = [
            (compound_label(db, k), target_preference(db, k, target["name"]))
            for k in potency["inchikey"]
        ]
        ratios = [(c, r) for c, r in ratios if r is not None]
        if not ratios:
            print("most selective: no compound has another comparable target to compare against")
        else:
            top_compound, top_ratio = max(ratios, key=lambda cr: cr[1])
            if top_ratio > 1:
                print(f"most selective: {top_compound}  ({top_ratio:.1f}-fold vs its next best target)")
            else:
                print(
                    f"most selective: none, really -- even {top_compound} is "
                    f"{1 / top_ratio:.1f}-fold more potent on a different target"
                )

    print()

243 of 2281 targets have at least one measurement
compounds tested
1    240
2      3
Name: targets, dtype: int64

showing 11 targets: the 10 with the most measurements on file, plus 1 more that have more than one compound tested

== Rattus norvegicus (protein) ==
compounds tested (1): CHEMBL1009
derived from: ChEMBL
most potent: no recognized potency measurement for this target

== Unchecked (protein) ==
compounds tested (2): CHEMBL100014, CHEMBL1009
derived from: ChEMBL
most potent (Potency, nM): CHEMBL1009  (12062.7 nM)
most selective: none, really -- even CHEMBL1009 is 3446.5-fold more potent on a different target

== Mus musculus (protein) ==
compounds tested (1): CHEMBL1009
derived from: ChEMBL
most potent (ED50, mg.kg-1): CHEMBL1009  (248 mg.kg-1)
most selective: no compound has another comparable target to compare against

== ADMET (protein) ==
compounds tested (1): CHEMBL1009
derived from: ChEMBL
most potent: no recognized potency measurement for this target

== No relevant tar

## Use case 3: per protein family

For a family target -- RAS, PARP 1/2/3, whatever a source has grouped -- which
chemical probes, chemogenomic compounds or drugs are available?

A family is just `target.type == "family"`; `target_uniprot` says which
accessions belong to it. `staging/use_cases`, the real data now loaded into
`probe.db`, does not have one: every one of its 2281 targets came in typed
`protein`, each with exactly one accession, and a good number of them are not
even proteins (`Homo sapiens`, `ADMET`, `No relevant target` show up as
"targets" too -- see the note at the top of this notebook). That is a gap in
this particular source, not in the schema or the code, so the cell below
checks for a family target and, finding none, falls back to
`staging/_template` -- the one place in this repository that does define a
family (`PARP 1, 2 and 3`) -- to show the same lookup actually running. The
same `target_profile` used in use case 2 works unchanged either way; only the
database it's pointed at differs.

One honesty note: the schema has no `probe` / `chemogenomic` / `drug` column
-- `compound` only carries an InChIKey, a SMILES and a name
(`database/schema.sql`). So "which compounds are available" is answered with
what actually is on file: an identifier, a SMILES and the potency on that
family, on whichever scale the fallback listing in use case 1/2 already
established.

In [4]:
families = db.table("target")
families = families[families.type == "family"]
families

,target_id,type,name


In [5]:
from loader import load
from loader.load import STAGING as STAGING_ROOT

if families.empty:
    print("No family-type targets in probe.db -- staging/use_cases loads everything as a "
          "single protein. Falling back to staging/_template to show the lookup working.\n")
    demo = ProbeDB(":memory:", create=True)
    load(demo, STAGING_ROOT / "_template", source="template")
    families = demo.table("target")
    families = families[families.type == "family"]
else:
    demo = db

FAMILY_NAME = families.iloc[0]["name"]  # swap in "RAS" or any other family name once one is loaded
family_id = int(families.iloc[0].target_id)

compound_keys, sources, scale, potency = target_profile(demo, family_id)

identity = demo.table("compound").merge(demo.table("chembl"), on="inchikey", how="left")
identity = identity[identity["inchikey"].isin(compound_keys)]

print(f"== {FAMILY_NAME} (family) ==")
print(f"compounds available ({len(compound_keys)}), derived from: {', '.join(sources)}")
print()

for _, row in identity.iterrows():
    chembl = row.chembl_id if pd.notna(row.chembl_id) else "no ChEMBL id on file"
    hit = potency[potency.inchikey == row.inchikey]
    if not hit.empty and scale:
        btype, unit = scale
        potency_str = f"{hit.value.iloc[0]:g} {unit} ({btype})"
    else:
        potency_str = "no comparable potency data"

    label = row["name"] or chembl
    print(label)
    print(f"  inchikey: {row.inchikey}")
    print(f"  chembl:   {chembl}")
    print(f"  smiles:   {row.smiles[:40]}...")
    print(f"  potency on {FAMILY_NAME}: {potency_str}")
    print()

No family-type targets in probe.db -- staging/use_cases loads everything as a single protein. Falling back to staging/_template to show the lookup working.

== PARP 1, 2 and 3 (family) ==
compounds available (1), derived from: literature

Olaparib
  inchikey: FDLYAMZZIXQODN-UHFFFAOYSA-N
  chembl:   CHEMBL521686
  smiles:   O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1...
  potency on PARP 1, 2 and 3: 20.89 nM (IC50)

